In [ ]:
DAY = 'day-1'

-- we need: all images from the same day -> one csv file representing records, another csv file representing data integrity for validation

In [ ]:
import os
import csv
from fractions import Fraction
from datetime import datetime
from PIL import Image
from PIL.ExifTags import TAGS, GPSTAGS
INPUT_DIR = "./images-day-wise/" + DAY + "/"
# Output files
OUTPUT_POINTS_CSV = os.path.join(INPUT_DIR, "0.points.csv")
OUTPUT_VALIDATION_CSV = os.path.join(INPUT_DIR, "1.check.csv")
REQUIRED_GPS = [
    "GPS GPSLatitude",
    "GPS GPSLatitudeRef",
    "GPS GPSLongitude",
    "GPS GPSLongitudeRef",
    "GPS GPSAltitude",
]
DATA_COLUMNS = [
    "file",
    "EXIF_READ_FAILED",
    "GPS GPSLatitude",
    "GPS GPSLatitudeRef",
    "GPS GPSLongitude",
    "GPS GPSLongitudeRef",
    "GPS GPSAltitude",
    "GPS_PARSE_FAILED",
    "EXIF DateTimeOriginal",
    "Latitude",
    "Longitude",
    "Altitude",
]
POINT_COLUMNS = [
    "slno",
    "datetime captured",
    "img name",
    "lat",
    "long",
    "altitude",
]
def rational_to_float(val):
    if isinstance(val, (list, tuple)):
        return sum(float(Fraction(str(v))) / (60 ** i) for i, v in enumerate(val))
    return float(Fraction(str(val)))
def get_exif(image_path):
    img = Image.open(image_path)
    raw = img._getexif()
    if not raw:
        return {}
    result = {}
    for tag_id, value in raw.items():
        tag = TAGS.get(tag_id, tag_id)
        if tag == "GPSInfo":
            for gps_id, gps_val in value.items():
                gps_tag = GPSTAGS.get(gps_id, gps_id)
                result[f"GPS {gps_tag}"] = gps_val
        else:
            result[f"EXIF {tag}"] = value
    return result
def parse_coords(exif):
    lat = rational_to_float(exif["GPS GPSLatitude"])
    lon = rational_to_float(exif["GPS GPSLongitude"])
    if exif.get("GPS GPSLatitudeRef", "N") != "N":
        lat = -lat
    if exif.get("GPS GPSLongitudeRef", "E") != "E":
        lon = -lon
    alt = float(Fraction(str(exif.get("GPS GPSAltitude", 0))))
    return lat, lon, alt
def parse_datetime(dt_str):
    if not dt_str:
        return None
    try:
        return datetime.strptime(dt_str, "%Y:%m:%d %H:%M:%S")
    except Exception:
        return None
def main():
    images = [
        f for f in os.listdir(INPUT_DIR)
        if f.lower().endswith((".jpg", ".jpeg"))
    ]
    valid_points = []
    data_rows = []
    for fname in sorted(images):
        row = {
            "file": fname,
            "EXIF_READ_FAILED": "missing",
            "GPS GPSLatitude": "missing",
            "GPS GPSLatitudeRef": "missing",
            "GPS GPSLongitude": "missing",
            "GPS GPSLongitudeRef": "missing",
            "GPS GPSAltitude": "missing",
            "GPS_PARSE_FAILED": "missing",
            "EXIF DateTimeOriginal": "missing",
            "Latitude": "",
            "Longitude": "",
            "Altitude": "",
        }
        fpath = os.path.join(INPUT_DIR, fname)
        try:
            exif = get_exif(fpath)
            row["EXIF_READ_FAILED"] = "x"
        except Exception:
            data_rows.append(row)
            continue
        for field in REQUIRED_GPS:
            if field in exif:
                row[field] = "x"
        dt = exif.get("EXIF DateTimeOriginal", "")
        if dt:
            row["EXIF DateTimeOriginal"] = "x"
        lat = "missing"
        lon = "missing"
        alt = "missing"

        if not any(f not in exif for f in REQUIRED_GPS):
            try:
                lat, lon, alt = parse_coords(exif)

                row["GPS_PARSE_FAILED"] = "x"
                row["Latitude"] = lat
                row["Longitude"] = lon
                row["Altitude"] = alt
            except Exception:
                pass

        data_rows.append(row)

        valid_points.append({
            "img_name": fname,
            "datetime": dt,
            "lat": lat,
            "lon": lon,
            "alt": alt,
            "dt_obj": parse_datetime(dt),
        })
    # Sort by capture datetime (earliest = 1)
    valid_points.sort(
        key=lambda x: (
            x["dt_obj"] is None,
            x["dt_obj"] if x["dt_obj"] else datetime.max,
        )
    )
    # Main points CSV
    with open(OUTPUT_POINTS_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=POINT_COLUMNS)
        writer.writeheader()
        for slno, point in enumerate(valid_points, start=1):
            writer.writerow({
                "slno": slno,
                "datetime captured": point["datetime"],
                "img name": point["img_name"],
                "lat": point["lat"],
                "long": point["lon"],
                "altitude": point["alt"],
            })
    # Validation CSV (same as original script)
    with open(OUTPUT_VALIDATION_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=DATA_COLUMNS)
        writer.writeheader()
        writer.writerows(data_rows)
    # print(f"Created: {OUTPUT_POINTS_CSV}")
    # print(f"Created: {OUTPUT_VALIDATION_CSV}")
    total_images = len(images)
    processed_points = len(valid_points)

    missing_lat_lon = sum(
        1 for point in valid_points
        if point["lat"] == "missing" or point["lon"] == "missing"
    )
    print(
        f"Summary | Total Images: {total_images} | "
        f"Points Created: {processed_points} | "
        f"Missing/Invalid Lat-Lon: {missing_lat_lon}"
    )
if __name__ == "__main__":
    main()

NORMALISATION:
every day - multiple images - 2 kmz - 2 csv

Approach 1:
day 1
    | images, all kml, all csv -- etc

day 2
    | images, all kml, all csv -- etc

Approach 2:
/img
    | day 1, day 2, day 3
/kml
    /kml - type 1
        | day 1, day 2, day 3
    /kml - type 2
        | day 1, day 2, day 3

field images -> csv     geographical data, validation csv [x] - KML not needed
form ->         csv     attribute data
|
|
|
final csv -> KML !!
geographical params | attribute params

In [ ]:
import os
import csv
from math import radians, sin, cos, sqrt, atan2
from datetime import datetime
POINTS_CSV = f"./images-day-wise/{DAY}/0.points.csv"
FIELD_CSV = f"./alive-web/data-csv/{DAY}.csv"
OUTPUT_CSV = f"./images-day-wise/{DAY}/2.merged.csv"
# Expected image captured after field submission
MAX_TIME_GAP_MIN = 2
# Expected GPS difference (meters)
MAX_DISTANCE_M = 100
def parse_datetime(dt_str):
    if not dt_str or dt_str == "missing":
        return None
    formats = [
        "%Y:%m:%d %H:%M:%S",
        "%Y-%m-%d %H:%M:%S",
        "%d-%m-%Y %H:%M:%S",
        "%d/%m/%Y %H:%M:%S",
    ]
    for fmt in formats:
        try:
            return datetime.strptime(dt_str.strip(), fmt)
        except Exception:
            pass
    return None
def haversine(lat1, lon1, lat2, lon2):
    r = 6371000
    lat1 = radians(float(lat1))
    lon1 = radians(float(lon1))
    lat2 = radians(float(lat2))
    lon2 = radians(float(lon2))
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = (
        sin(dlat / 2) ** 2
        + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    )
    return 2 * r * atan2(sqrt(a), sqrt(1 - a))
def main():
    with open(FIELD_CSV, newline="", encoding="utf-8") as f:
        field_rows = list(csv.DictReader(f))
    with open(POINTS_CSV, newline="", encoding="utf-8") as f:
        point_rows = list(csv.DictReader(f))
    n_field = len(field_rows)
    n_points = len(point_rows)
    print(f"Field records : {n_field}")
    print(f"Point records : {n_points}")
    if n_field != n_points:
        print(
            f"WARNING: Record count mismatch "
            f"({n_field} vs {n_points}). "
            f"Merging only first {min(n_field, n_points)} records."
        )
    merged_rows = []
    warning_count = 0
    merge_count = min(n_field, n_points)
    for idx in range(merge_count):
        field_row = field_rows[idx]
        point_row = point_rows[idx]
        warnings = []
        # ------------------------
        # Time validation
        # ------------------------
        spot_time = parse_datetime(field_row.get("spot_time", ""))
        image_time = parse_datetime(point_row.get("datetime captured", ""))
        if spot_time and image_time:
            gap_min = (image_time - spot_time).total_seconds() / 60
            if gap_min < 0:
                warnings.append(
                    f"image_before_submission({gap_min:.1f}min)"
                )
            elif gap_min > MAX_TIME_GAP_MIN:
                warnings.append(
                    f"large_time_gap({gap_min:.1f}min)"
                )
        # ------------------------
        # Spatial validation
        # ------------------------
        try:
            f_lat = field_row["latitude"]
            f_lon = field_row["longitude"]
            p_lat = point_row["lat"]
            p_lon = point_row["long"]
            if (
                f_lat not in ("", "missing")
                and f_lon not in ("", "missing")
                and p_lat not in ("", "missing")
                and p_lon not in ("", "missing")
            ):
                dist = haversine(
                    f_lat,
                    f_lon,
                    p_lat,
                    p_lon,
                )
                if dist > MAX_DISTANCE_M:
                    warnings.append(
                        f"gps_gap({dist:.1f}m)"
                    )
            else:
                dist = ""
        except Exception:
            dist = ""
            warnings.append("gps_validation_failed")
        if warnings:
            warning_count += 1
        merged = {
            "tree_id": field_row.get("tree_id", ""),
            "common_name": field_row.get("common_name", ""),
            "scientific_name": field_row.get("scientific_name", ""),
            "synonyms": field_row.get("synonyms", ""),
            "family": field_row.get("family", ""),
            "plant_type": field_row.get("plant_type", ""),
            "leaf_characteristics": field_row.get("leaf_characteristics", ""),
            "phenology": field_row.get("phenology", ""),
            "type": field_row.get("type", ""),
            "condition": field_row.get("condition", ""),
            "growth_stage": field_row.get("growth_stage", ""),
            "road_width_m": field_row.get("road_width_m", ""),
            "height_m": field_row.get("height_m", ""),
            "width_m": field_row.get("width_m", ""),

            "datetime captured": point_row.get("datetime captured", ""),
            "img name": point_row.get("img name", ""),
            "lat": point_row.get("lat", ""),
            "long": point_row.get("long", ""),
            "altitude": point_row.get("altitude", ""),

            "merge_index": idx + 1,
            "gps_distance_m": round(dist, 2) if dist != "" else "",
            "warning": "; ".join(warnings),
        }

        merged_rows.append(merged)
    if merged_rows:
        columns = [
            "tree_id",
            "common_name",
            "scientific_name",
            "taxa_id",
            "synonyms",
            "family",
            "leaf_characteristics",
            "phenology",
            "type",
            "plant_type",
            "condition",
            "growth_stage",
            "road_width_m",
            "height_m",
            "width_m",
            "datetime captured",
            "img name",
            "lat",
            "long",
            "altitude",
            "merge_index",
            "gps_distance_m",
            "warning",
        ]
        with open(
            OUTPUT_CSV,
            "w",
            newline="",
            encoding="utf-8",
        ) as f:
            writer = csv.DictWriter(
                f,
                fieldnames=columns,
                extrasaction="ignore",
            )
            writer.writeheader()
            writer.writerows(merged_rows)
    print()
    print("Merge Summary")
    print(f"Records merged : {merge_count}")
    print(f"Records flagged: {warning_count}")
    print(f"Output         : {OUTPUT_CSV}")
if __name__ == "__main__":
    main()

In [ ]:
import os
import pandas as pd
from IPython.display import display, clear_output, Image as IPyImage
import ipywidgets as widgets
CSV_PATH = f"./images-day-wise/{DAY}/2.merged.csv"
OUTPUT_PATH = f"./images-day-wise/{DAY}/2.merged.csv"
IMAGE_DIR = f"./images-day-wise/{DAY}"
FIELDS_TO_FILL = [
    "common_name",
    "scientific_name",
    "taxa_id",
    "synonyms",
    "family",
    "leaf_characteristics",
    "phenology",
    "type",
]
df = pd.read_csv(CSV_PATH, dtype=str).fillna("")
for col in FIELDS_TO_FILL:
    if col not in df.columns:
        df[col] = ""
    df[col] = df[col].astype(str)
current_idx = 0
title = widgets.HTML()
progress = widgets.HTML()
common_name = widgets.Text(description="Common")
scientific_name = widgets.Text(description="Scientific")
taxa_id = widgets.Text(description="Taxa ID")
synonyms = widgets.Textarea(description="Synonyms")
family = widgets.Text(description="Family")
leaf_characteristics = widgets.Textarea(description="Leaf")
phenology = widgets.Textarea(description="Phenology")
type_field = widgets.Text(description="Type")
save_btn = widgets.Button(description="Save & Next")
prev_btn = widgets.Button(description="Previous")
output = widgets.Output()
def load_record(idx):
    clear_output(wait=True)
    row = df.iloc[idx]
    title.value = f"<h3>Record {idx+1} / {len(df)}</h3>"
    progress.value = (
        f"<b>Tree ID:</b> {row.get('tree_id','')}<br>"
        f"<b>Image:</b> {row.get('img name','')}"
    )
    common_name.value = str(row.get("common_name", "") or "")
    scientific_name.value = str(row.get("scientific_name", "") or "")
    taxa_id.value = str(row.get("taxa_id", "") or "")
    synonyms.value = str(row.get("synonyms", "") or "")
    family.value = str(row.get("family", "") or "")
    leaf_characteristics.value = str(row.get("leaf_characteristics", "") or "")
    phenology.value = str(row.get("phenology", "") or "")
    type_field.value = str(row.get("type", "") or "")
    image_path = os.path.join(
        IMAGE_DIR,
        str(row["img name"])
    )
    display(title)
    display(progress)
    if os.path.exists(image_path):
        display(IPyImage(filename=image_path, width=700))
    else:
        print(f"Image not found: {image_path}")
    display(common_name)
    display(scientific_name)
    display(taxa_id)
    display(synonyms)
    display(family)
    display(leaf_characteristics)
    display(phenology)
    display(type_field)
    display(
        widgets.HBox(
            [prev_btn, save_btn]
        )
    )
def save_current():
    global current_idx
    df.loc[current_idx, "common_name"] = common_name.value
    df.loc[current_idx, "scientific_name"] = scientific_name.value
    df.loc[current_idx, "taxa_id"] = taxa_id.value
    df.loc[current_idx, "synonyms"] = synonyms.value
    df.loc[current_idx, "family"] = family.value
    df.loc[current_idx, "leaf_characteristics"] = leaf_characteristics.value
    df.loc[current_idx, "phenology"] = phenology.value
    df.loc[current_idx, "type"] = type_field.value
    df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")
def on_save_clicked(_):
    global current_idx
    save_current()
    if current_idx < len(df) - 1:
        current_idx += 1
    load_record(current_idx)
def on_prev_clicked(_):
    global current_idx
    save_current()
    if current_idx > 0:
        current_idx -= 1
    load_record(current_idx)
save_btn.on_click(on_save_clicked)
prev_btn.on_click(on_prev_clicked)
load_record(current_idx)

In [ ]:
import os
import csv
import zipfile
import xml.etree.ElementTree as ET
MERGED_CSV = os.path.join(INPUT_DIR, "2.merged.csv")
OUTPUT_KMZ = os.path.join(INPUT_DIR, f"{DAY}.kmz")
OUTPUT_KMZ_IMG = os.path.join(INPUT_DIR, f"{DAY}-img.kmz")
FIELDS = [
    "tree_id",
    "common_name",
    "scientific_name",
    "taxa_id",
    "synonyms",
    "family",
    "leaf_characteristics",
    "phenology",
    "type",
    "plant_type",
    "condition",
    "growth_stage",
    "road_width_m",
    "height_m",
    "width_m",
    "datetime captured",
    "img name",
    "lat",
    "long",
    "altitude",
]
def build_description(row, embed_image=False):
    html = "<![CDATA["
    for field in FIELDS:
        html += f"<b>{field}</b>: {row.get(field, '')}<br>"
    if embed_image:
        img_name = row.get("img name", "")
        html += f'<br><img src="images/{img_name}" width="800"/>'
    html += "]]>"
    return html
def build_kml(rows, embed_images=False):
    kml = ET.Element(
        "kml",
        xmlns="http://www.opengis.net/kml/2.2"
    )
    doc = ET.SubElement(kml, "Document")
    # ----------------------------
    # Path folder
    # ----------------------------
    path_folder = ET.SubElement(doc, "Folder")
    ET.SubElement(path_folder, "name").text = "Path"
    path_pm = ET.SubElement(path_folder, "Placemark")
    ET.SubElement(path_pm, "name").text = "1_to_N_Path"
    line = ET.SubElement(path_pm, "LineString")
    ET.SubElement(line, "tessellate").text = "1"
    coords = []
    for row in rows:
        lat = row.get("lat", "")
        lon = row.get("long", "")
        alt = row.get("altitude", "0")
        if lat in ("", "missing") or lon in ("", "missing"):
            continue
        try:
            coords.append(
                f"{float(lon)},{float(lat)},{float(alt or 0)}"
            )
        except Exception:
            pass
    ET.SubElement(line, "coordinates").text = " ".join(coords)
    # ----------------------------
    # Point folders
    # ----------------------------
    for row in rows:
        lat = row.get("lat", "")
        lon = row.get("long", "")
        if lat in ("", "missing") or lon in ("", "missing"):
            continue
        try:
            lat = float(lat)
            lon = float(lon)
            alt = float(row.get("altitude", 0) or 0)
        except Exception:
            continue
        tree_id = str(row.get("tree_id", ""))
        folder = ET.SubElement(doc, "Folder")
        ET.SubElement(folder, "name").text = tree_id
        pm = ET.SubElement(folder, "Placemark")
        ET.SubElement(pm, "name").text = tree_id
        ET.SubElement(pm, "description").text = build_description(
            row,
            embed_images
        )
        point = ET.SubElement(pm, "Point")
        ET.SubElement(
            point,
            "coordinates"
        ).text = f"{lon},{lat},{alt}"
    return ET.tostring(
        kml,
        encoding="unicode",
        xml_declaration=False
    )
def create_kmz(
    output_kmz,
    rows,
    embed_images=False
):
    kml_text = build_kml(
        rows,
        embed_images=embed_images
    )
    temp_kml = output_kmz.replace(".kmz", ".kml")
    with open(
        temp_kml,
        "w",
        encoding="utf-8"
    ) as f:
        f.write(
            '<?xml version="1.0" encoding="UTF-8"?>\n'
        )
        f.write(kml_text)
    with zipfile.ZipFile(
        output_kmz,
        "w",
        zipfile.ZIP_DEFLATED
    ) as zf:
        zf.write(
            temp_kml,
            arcname="doc.kml"
        )
        if embed_images:
            for row in rows:
                img_name = row.get("img name", "")
                img_path = os.path.join(
                    INPUT_DIR,
                    img_name
                )
                if os.path.exists(img_path):
                    zf.write(
                        img_path,
                        arcname=f"images/{img_name}"
                    )
    os.remove(temp_kml)
def main():
    with open(
        MERGED_CSV,
        newline="",
        encoding="utf-8"
    ) as f:
        rows = list(csv.DictReader(f))
    create_kmz(
        OUTPUT_KMZ,
        rows,
        embed_images=False
    )
    create_kmz(
        OUTPUT_KMZ_IMG,
        rows,
        embed_images=True
    )
    print(f"Created: {OUTPUT_KMZ}")
    print(f"Created: {OUTPUT_KMZ_IMG}")
    print(f"Records: {len(rows)}")
if __name__ == "__main__":
    main()